SQL 
```
WITH Values AS (
    SELECT c.customer_unique_id,
           MIN(CURRENT_DATE - o.order_purchase_timestamp::date) AS days_count,
           COUNT(DISTINCT o.order_id) AS orders_count,
           ROUND(SUM(i.price + i.freight_value)::numeric, 3) AS order_value
    FROM customers AS c
    JOIN orders AS o ON o.customer_id = c.customer_id
    JOIN order_items AS i ON o.order_id = i.order_id
    WHERE (o.order_status = 'delivered' OR o.order_status = 'shipped')
    GROUP BY c.customer_unique_id
),

Rating AS (
    SELECT v.customer_unique_id, v.days_count, v.orders_count,
    NTILE(5) OVER (ORDER BY v.days_count DESC) AS Recency,
    NTILE(5) OVER (ORDER BY v.orders_count) AS Frequency,
    NTILE(5) OVER (ORDER BY v.order_value) AS Monetary
    FROM Values AS v
)
SELECT
    customer_unique_id,
    Recency,
    Frequency,
    Monetary,
    CONCAT(Recency, Frequency, Monetary) AS RFM_Segment
FROM Rating
```

In [40]:
import numpy as np
from math import *

In [41]:
# coordinate example 50.01415009341115 (-90; 90), 36.31319462967358 (-180; 180)
rng = np.random.default_rng(seed=67)
lat_ton_min = [-90, -180]
lat_ton_max = [90, 180]
A = rng.uniform(low=lat_ton_min, high=lat_ton_max, size=(1000000, 2))
B = rng.uniform(low=lat_ton_min, high=lat_ton_max, size=(1000000, 2))

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2*R * asin(sqrt(a))
    
    return c
result = []
for i in range(1000000):
    lat1, lon1 = A[i]
    lat2, lon2 = B[i]
    distance = haversine_distance(lat1, lon1, lat2, lon2)
    result.append(distance)

%timeit [haversine_distance(A[i, 0], A[i, 1], B[i, 0], B[i, 1]) for i in range(1000000)]

1.46 s ± 27 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [42]:
def haversine_distance_numpy(coords1: np.ndarray, coords2: np.ndarray) -> np.ndarray:
    coords1 = np.radians(coords1)
    coords2 = np.radians(coords2)
    
    lat1, lon1 = coords1[:, 0], coords1[:, 1]
    lat2, lon2 = coords2[:, 0], coords2[:, 1]
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * 6371 * np.arcsin(np.sqrt(a))
    
    return c

distances = haversine_distance_numpy(A, B)
%timeit haversine_distance_numpy(A, B)

74.6 ms ± 2.18 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [75]:
import pandas as pd

In [44]:
orders_df = pd.read_csv(r'C:\Users\rodio\PycharmProjects\DataMining\src\olist\olist_orders_dataset.csv')
orders_df.head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00


In [45]:
order_items_df = pd.read_csv(r'C:\Users\rodio\PycharmProjects\DataMining\src\olist\olist_order_items_dataset.csv')
order_items_df.head(10)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69
6,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85
7,000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10 12:30:45,810.00,70.75
8,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65
9,0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06 14:10:56,53.99,11.40


In [46]:
products_df = pd.read_csv(r'C:\Users\rodio\PycharmProjects\DataMining\src\olist\olist_products_dataset.csv')
products_df.head(10)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
5,41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60.0,745.0,1.0,200.0,38.0,5.0,11.0
6,732bd381ad09e530fe0a5f457d81becb,cool_stuff,56.0,1272.0,4.0,18350.0,70.0,24.0,44.0
7,2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56.0,184.0,2.0,900.0,40.0,8.0,40.0
8,37cc742be07708b53a98702e77a21a02,eletrodomesticos,57.0,163.0,1.0,400.0,27.0,13.0,17.0
9,8c92109888e8cdf9d66dc7e463025574,brinquedos,36.0,1156.0,1.0,600.0,17.0,10.0,12.0


In [47]:
product_category_translation_df = pd.read_csv(r'C:\Users\rodio\PycharmProjects\DataMining\src\olist\product_category_name_translation.csv')
product_category_translation_df.head(10)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
5,esporte_lazer,sports_leisure
6,perfumaria,perfumery
7,utilidades_domesticas,housewares
8,telefonia,telephony
9,relogios_presentes,watches_gifts


In [48]:
combined_df = pd.merge(orders_df, order_items_df, how='left', on='order_id')
combined_df = pd.merge(combined_df, products_df, how='left', on='product_id')
combined_df['english_product_category_name'] = combined_df['product_category_name'].map(product_category_translation_df.set_index('product_category_name')['product_category_name_english'])
combined_df.head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,english_product_category_name
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,...,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,...,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,...,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,...,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,...,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00,1.0,060cb19345d90064d1015407193c233d,...,27.36,automotivo,49.0,608.0,1.0,7150.0,65.0,10.0,65.0,auto
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00,1.0,a1804276d9941ac0733cfd409f5206eb,...,16.05,NaN,NaN,NaN,NaN,600.0,35.0,35.0,15.0,NaN
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00,1.0,4520766ec412348b8d4caa5e8a18c464,...,15.17,automotivo,59.0,956.0,1.0,50.0,16.0,16.0,17.0,auto
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00,1.0,ac1789e492dcd698c5c10b97a671243a,...,16.05,moveis_decoracao,41.0,432.0,2.0,300.0,35.0,35.0,15.0,furniture_decor
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00,1.0,9a78fb9862b10749a117f7fc3c31f051,...,19.77,moveis_escritorio,45.0,527.0,1.0,9750.0,42.0,41.0,42.0,office_furniture


In [49]:
combined_df['order_timestamp'] = pd.to_datetime(combined_df['order_purchase_timestamp'])
combined_df['order_month'] = combined_df['order_timestamp'].dt.month
combined_df['order_year'] = combined_df['order_timestamp'].dt.year
combined_df.head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,english_product_category_name,order_timestamp,order_month,order_year
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,...,268.0,4.0,500.0,19.0,8.0,13.0,housewares,2017-10-02 10:56:33,10,2017
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,...,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,2018-07-24 20:41:37,7,2018
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,...,232.0,1.0,420.0,24.0,19.0,21.0,auto,2018-08-08 08:38:49,8,2018
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,...,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,2017-11-18 19:28:06,11,2017
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,...,316.0,4.0,250.0,51.0,15.0,15.0,stationery,2018-02-13 21:18:39,2,2018
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00,1.0,060cb19345d90064d1015407193c233d,...,608.0,1.0,7150.0,65.0,10.0,65.0,auto,2017-07-09 21:57:05,7,2017
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00,1.0,a1804276d9941ac0733cfd409f5206eb,...,NaN,NaN,600.0,35.0,35.0,15.0,NaN,2017-04-11 12:22:08,4,2017
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00,1.0,4520766ec412348b8d4caa5e8a18c464,...,956.0,1.0,50.0,16.0,16.0,17.0,auto,2017-05-16 13:10:30,5,2017
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00,1.0,ac1789e492dcd698c5c10b97a671243a,...,432.0,2.0,300.0,35.0,35.0,15.0,furniture_decor,2017-01-23 18:29:09,1,2017
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00,1.0,9a78fb9862b10749a117f7fc3c31f051,...,527.0,1.0,9750.0,42.0,41.0,42.0,office_furniture,2017-07-29 11:55:02,7,2017


In [50]:
combined_df.set_index(['order_year', 'order_month', 'english_product_category_name'], inplace=True)
combined_df

order_id  \
order_year order_month english_product_category_name                                     
2017       10          housewares                     e481f51cbdc54678b7cc49136f2d6af7   
2018       7           perfumery                      53cdb2fc8bc7dce0b6741e2150273451   
           8           auto                           47770eb9100c2d0c44946d9cf07ec65d   
2017       11          pet_shop                       949d5b44dbf5de918fe9c16f97b45f8a   
2018       2           stationery                     ad21c59c0840e6cb83a9ceb5573f8159   
...                                                                                ...   
                       baby                           63943bddc261676b46f01ca7ac2f7bd8   
2017       8           home_appliances_2              83c1379a015df1e13d02aae0204711ab   
2018       1           computers_accessories          11c177c8e97725db2631073c19f07b62   
                       computers_accessories          11c177c8e97725db2631073c19f07b62   
           3           health_beauty                  66dea50a8b16d9b4dee7af250b4be1a5   

                                                                           customer_id  \
order_year order_month english_product_category_name                                     
2017       10          housewares                     9ef432eb6251297304e76186b10a928d   
2018       7           perfumery                      b0830fb4747a6c6d20dea0b8c802d7ef   
           8           auto                           41ce2a54c0b03bf3443c3d931a367089   
2017       11          pet_shop                       f88197465ea7920adcdbec7375364d82   
2018       2           stationery                     8ab97904e6daea8866dbdbc4fb7aad2c   
...                                                                                ...   
                       baby                           1fca14ff2861355f6e5f14306ff977a7   
2017       8           home_appliances_2              1aa71eb042121263aafbe80c1b562c9c   
2018       1           computers_accessories          b331b74b18dc79bcdf6532d51e1637c1   
                       computers_accessories          b331b74b18dc79bcdf6532d51e1637c1   
           3           health_beauty                  edb027a75a1449115f6b43211ae02a24   

                                                     order_status  \
order_year order_month english_product_category_name                
2017       10          housewares                       delivered   
2018       7           perfumery                        delivered   
           8           auto                             delivered   
2017       11          pet_shop                         delivered   
2018       2           stationery                       delivered   
...                                                           ...   
                       baby                             delivered   
2017       8           home_appliances_2                delivered   
2018       1           computers_accessories            delivered   
                       computers_accessories            delivered   
           3           health_beauty                    delivered   

                                                     order_purchase_timestamp  \
order_year order_month english_product_category_name                            
2017       10          housewares                         2017-10-02 10:56:33   
2018       7           perfumery                          2018-07-24 20:41:37   
           8           auto                               2018-08-08 08:38:49   
2017       11          pet_shop                           2017-11-18 19:28:06   
2018       2           stationery                         2018-02-13 21:18:39   
...                                                                       ...   
                       baby                               2018-02-06 12:58:58   
2017       8           home_appliances_2                  2017-08-27 14:46:43   
2018       1           compu

In [53]:
df_unstacked = combined_df.groupby(['order_year', 'order_month', 'english_product_category_name'])[['price', 'freight_value']].sum().unstack(level='english_product_category_name')
df_unstacked.head(10)

price                   \
english_product_category_name agro_industry_and_commerce air_conditioning   
order_year order_month                                                      
2016       9                                         NaN              NaN   
           10                                        NaN          1707.09   
           12                                        NaN              NaN   
2017       1                                       65.97           663.70   
           2                                      224.84          3095.30   
           3                                       81.99          4103.81   
           4                                         NaN          2582.23   
           5                                     1579.94          1016.77   
           6                                     1390.00          2084.48   
           7                                     1180.00          1396.10   

                                                                       \
english_product_category_name      art arts_and_craftmanship    audio   
order_year order_month                                                  
2016       9                       NaN                   NaN      NaN   
           10                      NaN                   NaN   156.99   
           12                      NaN                   NaN      NaN   
2017       1                       NaN                   NaN      NaN   
           2                       NaN                   NaN   163.80   
           3                    223.25                   NaN  1134.77   
           4                    119.90                   NaN   942.20   
           5                   6967.65                 21.99  2081.18   
           6                    421.70                   NaN  1671.00   
           7                    386.75                129.90   771.69   

                                                                  \
english_product_category_name      auto      baby bed_bath_table   
order_year order_month                                             
2016       9                        NaN       NaN            NaN   
           10                   1833.25   1630.16         478.99   
           12                       NaN       NaN            NaN   
2017       1                    5218.53   6397.87        3960.16   
           2                   13162.40   3048.48       16282.73   
           3                   14482.07   3724.08       25773.02   
           4                   15548.17   3980.45       24347.69   
           5                   18640.03   9849.51       33346.45   
           6                   31370.69   7864.74       35114.81   
           7                   14119.74  15941.11       63888.75   

                                                                     ...  \
english_product_category_name books_general_interest books_imported  ...   
order_year order_month                                               ...   
2016       9                                     NaN            NaN  ...   
           10                                 119.50            NaN  ...   
           12                                    NaN            NaN  ...   
2017       1                                  234.89            NaN  ...   
           2                                  772.71          99.00  ...   
           3                                 2648.96            NaN  ...   
           4                                  923.38          19.99  ...   
           5                                 1841.57            NaN  ...   
           6                                 1940.08          99.00  ...   
           7                                 1292.40            NaN  ...   

                                      freight_value                         \
english_product_category_name security_and_services signaling_and_security   
order_year order_month                                               

In [81]:
price_bed_bath_table = df_unstacked.loc[((2018, [1,2,3]), ('price', 'bed_bath_table'))]
print(f"Bed bath table value on first quarter: \n{price_bed_bath_table}")

Bed bath table value on first quarter: 
order_year  order_month
2018        1              76377.79
            2              60690.88
            3              69256.39
Name: (price, bed_bath_table), dtype: float64


In [85]:
import plotly.express as px

df_plot = combined_df.groupby(['order_year', 'english_product_category_name'])['price'].sum().reset_index()

fig = px.line(
    df_plot,
    x='order_year',
    y='price',
    color='english_product_category_name',
    markers=True,
    title='Category Value Dynamics by Years',
    labels={
        'order_year': 'Year',
        'price': 'Total value',
        'english_product_category_name': 'Category'
    }
)

fig.update_layout(
    xaxis=dict(type='category'),
    hovermode='closest',
    template='plotly_white'
)

fig.show()